In [1]:
with open("train_utf8.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [2]:
len(text)

268049416

In [3]:
import os, time, random, unicodedata, regex as re
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------
# PART 0: Config & splits
# --------------------
TRAIN_PATH = "train_utf8.txt"        # A \t B per line
TEST_PATH  = "test_utf8.rand.txt"    # randomized pairs
SAVE_DIR   = Path("checkpoints"); SAVE_DIR.mkdir(parents=True, exist_ok=True)

# output files
meta_file  = SAVE_DIR / "meta.pt"    # tokenizer, merges, specials, block_size
ckpt_file  = SAVE_DIR / "lm.pt"      # model weights+config
OUT_PATH   = SAVE_DIR / "part1.txt"  # submission file

# Model / training (you can tweak)
batch_size    = 64
block_size    = 512      # ↑ longer context than 256
max_steps     = 20_000
eval_interval = 1000
learning_rate = 3e-4
n_embd        = 512
n_head        = 8
n_layer       = 8
dropout       = 0.2
seed          = 1337

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(seed); random.seed(seed)
use_amp = (device == "cuda")

# Special tokens (reserved ids)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
NUM_SPECIALS = 4

# GPT-2 style regex pretokenizer
gpt2pat = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

# Build A-side lines and split 80/10/10 by line order
A_lines = []
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if "\t" not in line: 
            continue
        a, _ = line.rstrip("\n").split("\t", 1)
        A_lines.append(a)

N = len(A_lines)
n_train = int(0.8 * N)
n_val   = int(0.1 * N)
train_lines = A_lines[:n_train]
val_lines   = A_lines[n_train:n_train+n_val]
test_lines  = A_lines[n_train+n_val:]
print(f"[splits] A-lines={N:,} | train={len(train_lines):,} | val={len(val_lines):,} | test={len(test_lines):,}")


[splits] A-lines=1,000,000 | train=800,000 | val=100,000 | test=100,000


Moving on to the tokenizer. This time we are implementing BPE to tokenize. This might not be the most efficient implementation but it gets the job done in a very simple manner. 

project/\
  tokenizer.py      # train/load BPE (with progress), encode/decode\
  model.py          # transformer blocks + GPTLanguageModel\
  data.py           # load A-side, split 80/10/10, build streams, batching\
  train.py          # trains the LM, logs, saves lm.pt\
  score.py          # avg_nll_string + write part1.txt\
  generate.py       # generate_text sampler (temp/top-k/top-p)\
  checkpoints/      # meta.pt (tokenizer), lm.pt (weights), part1.txt


In [4]:
%run tokenizer.py

In [5]:
%run model.py

In [6]:
%run data.py

OOM Error:%run train.py

[splits] train=800,000 val=100,000 test=100,000\
[BPE(50k-rand)] start: stream_len=6,732,200 target_merges=3,836\
[BPE(50k-rand)] merges=   191/3836   (  5.0%) max_pair_freq=3828     stream_len=3,318,406 elapsed= 146.6s\
[BPE(50k-rand)] merges=   382/3836   ( 10.0%) max_pair_freq=1707     stream_len=2,843,274 elapsed= 268.5s\
[BPE(50k-rand)] merges=   573/3836   ( 14.9%) max_pair_freq=1042     stream_len=2,589,444 elapsed= 392.9s\
[BPE(50k-rand)] merges=   764/3836   ( 19.9%) max_pair_freq=762      stream_len=2,420,082 elapsed= 506.2s\
[BPE(50k-rand)] merges=   955/3836   ( 24.9%) max_pair_freq=590      stream_len=2,293,391 elapsed= 630.0s\
[BPE(50k-rand)] merges=  1146/3836   ( 29.9%) max_pair_freq=481      stream_len=2,191,719 elapsed= 760.7s\
[BPE(50k-rand)] merges=  1337/3836   ( 34.9%) max_pair_freq=393      stream_len=2,109,021 elapsed= 889.8s\
[BPE(50k-rand)] merges=  1528/3836   ( 39.8%) max_pair_freq=332      stream_len=2,039,944 elapsed=1004.0s\
[BPE(50k-rand)] merges=  1719/3836   ( 44.8%) max_pair_freq=287      stream_len=1,981,184 elapsed=1115.6s\
[BPE(50k-rand)] merges=  1910/3836   ( 49.8%) max_pair_freq=251      stream_len=1,929,966 elapsed=1228.9s\
[BPE(50k-rand)] merges=  2101/3836   ( 54.8%) max_pair_freq=225      stream_len=1,884,677 elapsed=1341.4s\
[BPE(50k-rand)] merges=  2292/3836   ( 59.7%) max_pair_freq=203      stream_len=1,843,946 elapsed=1460.0s\
[BPE(50k-rand)] merges=  2483/3836   ( 64.7%) max_pair_freq=183      stream_len=1,807,179 elapsed=1582.7s\
[BPE(50k-rand)] merges=  2674/3836   ( 69.7%) max_pair_freq=166      stream_len=1,773,940 elapsed=1718.5s\
[BPE(50k-rand)] merges=  2865/3836   ( 74.7%) max_pair_freq=153      stream_len=1,743,537 elapsed=1836.1s\
[BPE(50k-rand)] merges=  3056/3836   ( 79.7%) max_pair_freq=139      stream_len=1,715,607 elapsed=1953.7s\
[BPE(50k-rand)] merges=  3247/3836   ( 84.6%) max_pair_freq=128      stream_len=1,690,174 elapsed=2075.5s\
[BPE(50k-rand)] merges=  3438/3836   ( 89.6%) max_pair_freq=118      stream_len=1,666,709 elapsed=2187.8s\
[BPE(50k-rand)] merges=  3629/3836   ( 94.6%) max_pair_freq=110      stream_len=1,644,986 elapsed=2303.9s\
[BPE(50k-rand)] merges=  3820/3836   ( 99.6%) max_pair_freq=103      stream_len=1,624,655 elapsed=2440.6s\
[BPE(50k-rand)] merges=  3836/3836   (100.0%) max_pair_freq=103      stream_len=1,623,007 elapsed=2452.5s\
[BPE(50k-rand)] done: total_merges=3,836 raw_vocab_size=4092 time=2452.5s\
[tokenizer] trained & saved checkpoints/meta.pt\
[streams] lens: 28602706 3579951 3577602\
[model] params=27.57M

---------------------------------------------------------------------------\
OutOfMemoryError                          Traceback (most recent call last)\
File /notebooks/TokenizedLM/train.py:75\
     73 xb,yb = get_batch(train_stream, batch_size, block_size, device)\
     74 with torch.cuda.amp.autocast(enabled=use_amp):\
---> 75     _, loss = model(xb, yb, ignore_index=PAD_ID)\
     76 optimizer.zero_grad(set_to_none=True)\
     77 scaler.scale(loss).backward()

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacty of 15.72 GiB of which 55.44 MiB is free. Process 2050660 has 15.66 GiB memory in use. Of the allocated memory 15.40 GiB is allocated by PyTorch, and 67.67 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

I OOM’d because the new config is quite a bit heavier: n_embd=512, n_head=8, n_layer=8 and block_size=512 (attention memory scales ~O(B · T² · n_layer)). With BPE my average T per window is still 512 during training, so memory blows up fast.

[splits] train=800,000 val=100,000 test=100,000\
[tokenizer] loaded checkpoints/meta.pt\
[streams] lens: 28602706 3579951 3577602\
[model] params=12.31M\
step      1 | train 8.1749 | val 8.1735 | test 8.1746 | 0.2 min\
step   1000 | train 5.0925 | val 5.0864 | test 5.0958 | 12.4 min\
step   2000 | train 4.8278 | val 4.8323 | test 4.8302 | 24.5 min\
step   3000 | train 4.7214 | val 4.7180 | test 4.7162 | 36.5 min\
step   4000 | train 4.6183 | val 4.6336 | test 4.6160 | 48.5 min\
step   5000 | train 4.5633 | val 4.5632 | test 4.5648 | 60.6 min\
step   6000 | train 4.5293 | val 4.5226 | test 4.5079 | 72.6 min\
step   7000 | train 4.4853 | val 4.4994 | test 4.5053 | 84.6 min\
step   8000 | train 4.4827 | val 4.4844 | test 4.4895 | 97.0 min\
step   9000 | train 4.4695 | val 4.4750 | test 4.4801 | 109.1 min\
step  10000 | train 4.4754 | val 4.4879 | test 4.4736 | 121.3 min\
[ckpt] saved checkpoints/lm.pt


In [5]:
%run train.py

[splits] train=800,000 val=100,000 test=100,000
[tokenizer] loaded checkpoints/meta.pt


KeyboardInterrupt: 

[splits] train=800,000 val=100,000 test=100,000\
[tokenizer] loaded checkpoints/meta.pt\
[streams] lens: 28602706 3579951 3577602\
[model] params=12.31M\
step      1 | train 8.1749 | val 8.1735 | test 8.1746 | 0.1 min\
step   1000 | train 5.0925 | val 5.0864 | test 5.0958 | 11.9 min\
step   2000 | train 4.8278 | val 4.8323 | test 4.8302 | 24.0 min\
step   3000 | train 4.7214 | val 4.7180 | test 4.7162 | 35.9 min\
step   4000 | train 4.6183 | val 4.6336 | test 4.6160 | 48.1 min\
step   5000 | train 4.5633 | val 4.5632 | test 4.5648 | 60.1 min\
step   6000 | train 4.5293 | val 4.5226 | test 4.5079 | 72.1 min\
step   7000 | train 4.4853 | val 4.4994 | test 4.5053 | 84.0 min\
step   8000 | train 4.4827 | val 4.4844 | test 4.4895 | 95.9 min\
step   9000 | train 4.4695 | val 4.4750 | test 4.4801 | 107.8 min\
step  10000 | train 4.4754 | val 4.4879 | test 4.4736 | 120.1 min\
step  11000 | train 4.4889 | val 4.4904 | test 4.4929 | 132.0 min\
step  12000 | train 4.5024 | val 4.5082 | test 4.5181 | 143.8 min\
step  13000 | train 4.5238 | val 4.5248 | test 4.5224 | 155.6 min\
step  14000 | train 4.5290 | val 4.5444 | test 4.5404 | 167.3 min\
step  15000 | train 4.5619 | val 4.5652 | test 4.5604 | 179.1 min\
step  16000 | train 4.5816 | val 4.5729 | test 4.5751 | 190.9 min\
step  17000 | train 4.5730 | val 4.5632 | test 4.5768 | 202.7 min\
step  18000 | train 4.5731 | val 4.5822 | test 4.5727 | 214.6 min\
step  19000 | train 4.5941 | val 4.6029 | test 4.5995 | 226.5 min\
step  20000 | train 4.6209 | val 4.6158 | test 4.6306 | 238.3 min\
[ckpt] saved checkpoints/lm.pt


In [6]:
%run score.py

[loaded] tokenizer+model
[classify] scoring test set and writing checkpoints/part1.txt


KeyboardInterrupt: 

In [7]:
%run generate.py


In conclusion, I believe that it will make it possible as far more precisely support.We should also understressive and European citizens are unemployment and and security and the rule of law.I cannot look at that these issues should not have thirdly.Let us we tookind the ped toys and that he was stying the biggest , just a vacher father .There are our report by Mr Vinebrington and Mr Sán Chaband-Söt, at st, on the Commission concerning its own national Fiterce of the He on behalf of the Committee on FRural Development (ATREE/ET) F SLAT on the T).A dad is a secision process in relation to find a group of Denjoya .Tok is unny at aver the Suno , 


In [8]:
# ============================================================
# Sanity check on train_utf8.txt (TEST = last 10% of pairs)
# Uses trained `model` + `avg_nll_string` (BPE-aware).
# Prints progress every 1/10th of the evaluated inputs.
# ============================================================

import random, math, torch, time

# Config
SPLIT_RATIO    = (0.8, 0.1, 0.1)  # train/val/test by line order
MAX_EVAL       = 20000            # set to None to evaluate all test pairs
TIE_BIAS_TO_A  = True             # in exact ties, count as A (gold=A)

assert 'TRAIN_PATH' in globals(), "TRAIN_PATH not found."
assert 'avg_nll_string' in globals(), "avg_nll_string not found."
assert 'model' in globals(), "model not found. Train or load the model first."

# Ensure eval mode + no grad
model.eval()
torch.set_grad_enabled(False)

# Read all pairs from train file
pairs = []
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if "\t" not in line:
            continue
        a, b = line.rstrip("\n").split("\t", 1)
        pairs.append((a, b))

N = len(pairs)
n_train = int(SPLIT_RATIO[0] * N)
n_val   = int(SPLIT_RATIO[1] * N)
test_pairs = pairs[n_train + n_val:]  # final 10%

# Optionally subsample for speed
idxs = list(range(len(test_pairs)))
random.seed(1337)
random.shuffle(idxs)
if MAX_EVAL is not None:
    idxs = idxs[:MAX_EVAL]

correct = 0
pred_A = 0
ties = 0
tot = len(idxs)
mistakes = []  # (margin, a, b, nlla, nllb)

# progress cadence: print ~10 times
progress_every = max(1, math.ceil(tot / 10))
t0 = time.time()

for j, k in enumerate(idxs, 1):
    a, b = test_pairs[k]
    nll_a = avg_nll_string(model, a)
    nll_b = avg_nll_string(model, b)

    if abs(nll_a - nll_b) < 1e-12:
        ties += 1
        pred = 'A' if TIE_BIAS_TO_A else 'B'
    else:
        pred = 'A' if nll_a < nll_b else 'B'

    if pred == 'A':
        pred_A += 1
        correct += 1   # gold is A for train file pairs
    else:
        margin = nll_b - nll_a   # negative => model prefers B
        mistakes.append((margin, a, b, nll_a, nll_b))

    # progress print every 1/10th (and at the very end)
    if (j % progress_every == 0) or (j == tot):
        pct = 100.0 * j / tot
        acc_so_far = correct / j
        predA_so_far = pred_A / j
        elapsed = time.time() - t0
        print(f"[{j:,}/{tot:,} | {pct:5.1f}%] "
              f"acc={acc_so_far*100:5.2f}%  A%={predA_so_far*100:5.2f}%  ties={ties}  "
              f"elapsed={elapsed:5.1f}s",
              flush=True)

# Final stats
acc = (correct / tot) if tot else 0.0
predA_pct = (pred_A / tot) if tot else 0.0

print(f"\nTest pairs evaluated: {tot} (from {N} total; last {SPLIT_RATIO[2]*100:.0f}% of lines)")
print(f"Accuracy (gold=A): {acc*100:.2f}%")
print(f"Predicted 'A' fraction: {predA_pct*100:.2f}%")
print(f"Ties: {ties}")

# Show a few mistakes where the model preferred B
mistakes.sort(key=lambda x: x[0])  # most negative margin first
show = min(5, len(mistakes))
if show:
    print("\nTop mistakes (model preferred B):")
    for i in range(show):
        margin, a, b, nlla, nllb = mistakes[i]
        print(f"\n#{i+1} margin={margin:.4f} (nll_a={nlla:.4f}, nll_b={nllb:.4f})")
        print("A:", a)
        print("B:", b)
else:
    print("\nNo mistakes in the evaluated subset 🎉")


[2,000/20,000 |  10.0%] acc=92.00%  A%=92.00%  ties=0  elapsed= 38.0s
[4,000/20,000 |  20.0%] acc=91.83%  A%=91.83%  ties=0  elapsed= 75.6s
[6,000/20,000 |  30.0%] acc=91.95%  A%=91.95%  ties=0  elapsed=113.8s
[8,000/20,000 |  40.0%] acc=91.76%  A%=91.76%  ties=0  elapsed=152.0s
[10,000/20,000 |  50.0%] acc=91.94%  A%=91.94%  ties=0  elapsed=190.2s
[12,000/20,000 |  60.0%] acc=91.78%  A%=91.78%  ties=0  elapsed=228.2s
[14,000/20,000 |  70.0%] acc=91.72%  A%=91.72%  ties=0  elapsed=266.2s
[16,000/20,000 |  80.0%] acc=91.73%  A%=91.73%  ties=0  elapsed=303.7s
[18,000/20,000 |  90.0%] acc=91.69%  A%=91.69%  ties=0  elapsed=341.5s
[20,000/20,000 | 100.0%] acc=91.64%  A%=91.64%  ties=0  elapsed=379.3s

Test pairs evaluated: 20000 (from 1000000 total; last 10% of lines)
Accuracy (gold=A): 91.64%
Predicted 'A' fraction: 91.64%
Ties: 0

Top mistakes (model preferred B):

#1 margin=-1.5118 (nll_a=4.5390, nll_b=3.0271)
A: I welcome the Garosci report.
B: I welcome the Greece report.

#2 margin=-

[2,000/20,000 |  10.0%] acc=90.90%  A%=90.90%  ties=0  elapsed= 35.4s\
[4,000/20,000 |  20.0%] acc=90.90%  A%=90.90%  ties=0  elapsed= 71.2s\
[6,000/20,000 |  30.0%] acc=90.82%  A%=90.82%  ties=0  elapsed=109.3s\
[8,000/20,000 |  40.0%] acc=90.69%  A%=90.69%  ties=0  elapsed=145.5s\
[10,000/20,000 |  50.0%] acc=90.96%  A%=90.96%  ties=0  elapsed=183.1s\
[12,000/20,000 |  60.0%] acc=90.84%  A%=90.84%  ties=0  elapsed=220.8s\
[14,000/20,000 |  70.0%] acc=90.89%  A%=90.89%  ties=0  elapsed=258.6s\
[16,000/20,000 |  80.0%] acc=90.86%  A%=90.86%  ties=0  elapsed=296.0s\
[18,000/20,000 |  90.0%] acc=90.86%  A%=90.86%  ties=0  elapsed=333.8s\
[20,000/20,000 | 100.0%] acc=90.83%  A%=90.83%  ties=0  elapsed=370.6s\

Test pairs evaluated: 20000 (from 1000000 total; last 10% of lines)\
Accuracy (gold=A): 90.83%\
Predicted 'A' fraction: 90.83%\
Ties: 0

Top mistakes (model preferred B):

#1 margin=-1.3545 (nll_a=4.5603, nll_b=3.2058)\
A: I welcome the Garosci report.\
B: I welcome the Greece report.

#2 margin=-1.2351 (nll_a=4.3974, nll_b=3.1623)\
A: It is not a ‘Lamfalussy’ directive.\
B: It is not a integrated directive.

#3 margin=-1.0709 (nll_a=5.5625, nll_b=4.4917)\
A: Road Transport\
B: On Transport

#4 margin=-0.8836 (nll_a=5.3401, nll_b=4.4565)\
A: everything stops yeah\
B: everythings tops yeah

#5 margin=-0.7493 (nll_a=5.0538, nll_b=4.3045)\
A: Click again.\
B: Crick again.


In [4]:
generate_text("In four score and seven years", stream=True)

In four score and seven yearsI elope with of the European China strange prime burd doced with the grances for choice services it Sercompanies there were inventited.Air beyale related out for happening and Sgen poweringly explic enterable.Lord tracks as well as soon powers what commodes to leave him down .We are have been sulapply that threaten that the review and internal vicether powers of our strategy in the European Union'Audlian in, and by our lifel, boate the governments for problems.Althezonia, we wonder cosetary cohesion that them (Enian organ technologies are effective and Belgical Treaty in the plan, and I from this Christian parties, have contacts trade comstate coordination which ress it is deocate only doubtes that organized is the breashed.But once after doa time for the gang co.


"In four score and seven yearsI elope with of the European China strange prime burd doced with the grances for choice services it Sercompanies there were inventited.Air beyale related out for happening and Sgen poweringly explic enterable.Lord tracks as well as soon powers what commodes to leave him down .We are have been sulapply that threaten that the review and internal vicether powers of our strategy in the European Union'Audlian in, and by our lifel, boate the governments for problems.Althezonia, we wonder cosetary cohesion that them (Enian organ technologies are effective and Belgical Treaty in the plan, and I from this Christian parties, have contacts trade comstate coordination which ress it is deocate only doubtes that organized is the breashed.But once after doa time for the gang co."

In [1]:
from pathlib import Path
import shutil

def copy_everything(src, dst, overwrite=True, keep_symlinks=False, ignore=None):
    src = Path(src).resolve()
    dst = Path(dst).resolve()

    print("CWD: ", Path.cwd())
    print("SRC:", src, "| exists?", src.is_dir())
    print("DST:", dst)

    if not src.is_dir():
        raise FileNotFoundError(f"Source directory not found: {src}")

    # ensure destination's parent exists
    dst.parent.mkdir(parents=True, exist_ok=True)

    kwargs = {
        "dirs_exist_ok": overwrite,   # needs Python 3.8+
        "symlinks": keep_symlinks,    # True = keep symlinks, False = copy targets
    }
    if ignore:
        from shutil import ignore_patterns
        kwargs["ignore"] = ignore_patterns(*ignore)

    shutil.copytree(src, dst, **kwargs)

    # show a quick listing
    print("\nCopied. Top-level in DST:")
    for p in sorted(dst.iterdir()):
        try:
            size = f"{p.stat().st_size/1e6:.2f} MB" if p.is_file() else "<dir>"
        except Exception:
            size = "?"
        print(" -", p.name, size)

# Example: copy the entire 'checkpoints' tree into './artifacts'
copy_everything("checkpoints", "artifacts", overwrite=True)


CWD:  /notebooks/TokenizedLM
SRC: /notebooks/TokenizedLM/checkpoints | exists? True
DST: /notebooks/TokenizedLM/artifacts

Copied. Top-level in DST:
 - lm.pt 58.75 MB
 - meta.pt 0.06 MB
 - part1.txt 0.20 MB


In [12]:
print(avg_nll_string(model, "This is an English sentence.")  )
print(avg_nll_string(model, "Th1s is n0t gr8 Engl1sh!!!")  )  


3.6710774898529053
8.183836873372396
